# 交通标志检测与识别系统 - 交互式演示
基于 YOLOv8 + GTSDB 数据集

## 1. 环境检查

In [ ]:
import torch
import ultralytics
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Ultralytics: {ultralytics.__version__}')

## 2. 加载模型

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'code'))
from model import TrafficSignDetector

detector = TrafficSignDetector()

# 选择要加载的模型权重
results_dir = os.path.join(os.getcwd(), '..', 'results')
weight_path = os.path.join(results_dir, 'improved_best.pt')

if not os.path.exists(weight_path):
    weight_path = os.path.join(results_dir, 'baseline_best.pt')

if os.path.exists(weight_path):
    print(f'加载模型: {weight_path}')
else:
    print('警告: 未找到训练好的模型权重，请先运行 train.py')
    print(f'查找路径: {weight_path}')

## 3. 单张图片推理

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random
from glob import glob

CLASS_NAMES = ['speed_limit', 'no_entry', 'direction', 'crosswalk', 'stop_yield']
CLASS_CN = ['限速', '禁止通行', '直行/转弯', '人行横道', '停车让行']
COLORS = [(0, 255, 0), (0, 0, 255), (255, 0, 0), (255, 255, 0), (0, 255, 255)]

def draw_boxes(img, results):
    """在图片上绘制检测框"""
    img_copy = img.copy()
    if results[0].boxes is not None:
        boxes = results[0].boxes
        for i in range(len(boxes)):
            x1, y1, x2, y2 = map(int, boxes.xyxy[i].tolist())
            cls_id = int(boxes.cls[i])
            conf = boxes.conf[i].item()
            color = COLORS[cls_id]
            cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, 2)
            label = f'{CLASS_CN[cls_id]} {conf:.2f}'
            cv2.putText(img_copy, label, (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img_copy

# 随机选一张验证集图片
val_dir = os.path.join(os.getcwd(), '..', 'dataset', 'yolo_format', 'images', 'val')
if os.path.exists(val_dir):
    img_files = glob(os.path.join(val_dir, '*.jpg'))
    if img_files:
        test_img = random.choice(img_files)
        print(f'测试图片: {test_img}')
        
        if os.path.exists(weight_path):
            results = detector.predict(test_img, weight_path, conf=0.3)
            img = cv2.imread(test_img)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_vis = draw_boxes(img, results)
            
            plt.figure(figsize=(10, 8))
            plt.imshow(img_vis)
            plt.axis('off')
            plt.title(f'Detection Results ({len(results[0].boxes)} objects)')
            plt.show()
        else:
            img = cv2.imread(test_img)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 8))
            plt.imshow(img)
            plt.axis('off')
            plt.title('Sample Image (no model loaded)')
            plt.show()
    else:
        print('验证集为空')
else:
    print('请先运行数据预处理')

## 4. 多图批量推理

In [ ]:
def show_batch_results(img_dir, num_samples=6):
    """批量展示推理结果"""
    if not os.path.exists(weight_path):
        print('未找到模型权重')
        return
    
    img_files = glob(os.path.join(img_dir, '*.jpg'))
    if len(img_files) == 0:
        print('目录下无图片')
        return
    
    samples = random.sample(img_files, min(num_samples, len(img_files)))
    
    rows = (num_samples + 2) // 3
    fig, axes = plt.subplots(rows, 3, figsize=(15, 5*rows))
    axes = axes.flatten()
    
    for i, img_path in enumerate(samples):
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = detector.predict(img_path, weight_path, conf=0.3)
        img_vis = draw_boxes(img, results)
        axes[i].imshow(img_vis)
        axes[i].axis('off')
        n_objs = len(results[0].boxes) if results[0].boxes else 0
        axes[i].set_title(f'{os.path.basename(img_path)} ({n_objs} detections)')
    
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

if os.path.exists(val_dir) and os.path.exists(weight_path):
    show_batch_results(val_dir, num_samples=6)
else:
    print('请先完成数据预处理和模型训练')

## 5. 场景对比分析
对比不同条件（明亮/正常/低光/模糊）下的检测效果

In [ ]:
from data_utils import classify_scene, OUR_CLASS_NAMES

def analyze_by_scene():
    """按场景分类展示检测效果"""
    if not os.path.exists(val_dir):
        print('验证集不存在')
        return
    
    img_files = glob(os.path.join(val_dir, '*.jpg'))
    scene_groups = {'bright': [], 'normal': [], 'dark': [], 'blurry': []}
    for f in img_files:
        s = classify_scene(f)
        scene_groups[s].append(f)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    scene_cn = {'bright': '明亮', 'normal': '正常', 'dark': '低光/夜间', 'blurry': '模糊'}
    
    for idx, (scene, files) in enumerate(scene_groups.items()):
        ax = axes[idx // 2, idx % 2]
        if not files:
            ax.text(0.5, 0.5, f'{scene_cn[scene]}: 无样本', ha='center', va='center')
            ax.set_title(scene_cn[scene])
            ax.axis('off')
            continue
        
        sample = random.choice(files)
        img = cv2.imread(sample)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if os.path.exists(weight_path):
            results = detector.predict(sample, weight_path, conf=0.3)
            img = draw_boxes(img, results)
        
        ax.imshow(img)
        ax.set_title(f'{scene_cn[scene]} ({len(files)} 张)')
        ax.axis('off')
    
    plt.suptitle('不同场景条件检测效果对比', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

analyze_by_scene()

## 6. 模型性能对比
Baseline vs Improved 模型

In [ ]:
import json

comparison_path = os.path.join(results_dir, 'comparison.json')
if os.path.exists(comparison_path):
    with open(comparison_path) as f:
        comp = json.load(f)
    
    metrics = ['mAP50', 'mAP50_95', 'precision', 'recall', 'f1']
    labels_cn = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'F1-Score']
    
    baseline_vals = [comp.get('baseline', {}).get(m, 0) for m in metrics]
    improved_vals = [comp.get('improved', {}).get(m, 0) for m in metrics]
    
    x = range(len(metrics))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar([i - width/2 for i in x], baseline_vals, width, label='Baseline', color='#3498db')
    bars2 = ax.bar([i + width/2 for i in x], improved_vals, width, label='Improved', color='#e74c3c')
    
    ax.set_ylabel('Score')
    ax.set_title('Baseline vs Improved Model Performance')
    ax.set_xticks(x)
    ax.set_xticklabels(labels_cn)
    ax.legend()
    ax.set_ylim(0, 1)
    
    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.3f}', ha='center', fontsize=9)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.3f}', ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
else:
    print('请先运行 eval.py 生成 comparison.json')

## 7. ONNX 模型导出 (加分项)

In [ ]:
def export_to_onnx():
    if not os.path.exists(weight_path):
        print('未找到模型权重')
        return
    
    onnx_path = os.path.join(results_dir, 'model.onnx')
    print(f'导出 ONNX 模型到: {onnx_path}')
    result = detector.export_onnx(weight_path, onnx_path)
    print(f'ONNX 模型已导出: {result}')
    
    # 检查文件大小
    size_mb = os.path.getsize(onnx_path) / 1024 / 1024
    print(f'模型大小: {size_mb:.1f} MB')

# 取消注释以导出
# export_to_onnx()
print('ONNX 导出代码已就绪，取消注释上面的 export_to_onnx() 即可运行')